### **1. Environment Setup & Authentication**

In [ ]:
# Install required libraries
!pip install -q osmnx geopandas folium earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 8.4 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import geopandas as gpd
import folium
import ee
import networkx as nx

In [ ]:
# Trigger the authentication flow for Google Earth Engine
ee.Authenticate()
ee.Initialize(project='gdsexp9') # Replace with your GEE project ID

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


### **2. Defining the AOI and Fetching Urban Services**

In [ ]:
# Define the Area of Interest (AOI)
place_name = "Andheri, Mumbai, Maharashtra, India"

# Fetch the boundary of Andheri
andheri_boundary = ox.geocode_to_gdf(place_name)
aoi_geometry = andheri_boundary.geometry.iloc[0]

# Since Andheri boundary might not cover all features directly in OSM administrative queries,
# we can fetch features within a buffer distance around the center centroid.
center_point = (aoi_geometry.centroid.y, aoi_geometry.centroid.x)
tags = {"amenity": ["hospital", "school"], "leisure": ["park"]}

# Fetch amenities within 3000 meters of Andheri's center
amenities = ox.features_from_point(center_point, tags=tags, dist=3000)

# Separate points for mapping (ignoring polygons for simple markers)
amenities_points = amenities[amenities.geom_type == 'Point']

# Set center coordinates for mapping in subsequent steps
center_lat, center_lon = center_point[0], center_point[1]

print(f"Fetched {len(amenities_points)} service locations near Andheri center.")

Fetched 130 service locations near Andheri center.


### **3. Road Network Analysis**

In [ ]:
# Fetch the street network for driving using a buffer around the center point
# because Andheri is represented as a point location rather than a polygon boundary in OSM
G = ox.graph_from_point(center_point, dist=2000, network_type="drive")

# Calculate basic network metrics (e.g., node centrality to find busy intersections)
# Note: For a large area, betweenness centrality can be computationally heavy.
# We'll calculate it for a simplified graph or just plot the raw network.
G_projected = ox.project_graph(G)
nodes, edges = ox.graph_to_gdfs(G_projected)

print(f"Network extracted: {len(nodes)} nodes and {len(edges)} edges.")

Network extracted: 1940 nodes and 4286 edges.


### **4. Earth Engine: NDVI & Environmental Overlays**

In [ ]:
# Convert bounding box to Earth Engine format
bbox = andheri_boundary.total_bounds # [minx, miny, maxx, maxy]
ee_bounds = ee.Geometry.Rectangle([bbox[0], bbox[1], bbox[2], bbox[3]])

# Fetch Sentinel-2 Surface Reflectance data (Filter by date and cloud cover)
s2 = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(ee_bounds) \
    .filterDate('2023-01-01', '2023-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median() \
    .clip(ee_bounds)

# Calculate NDVI: (NIR - Red) / (NIR + Red) -> (B8 - B4) / (B8 + B4)
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Define visualization parameters for Earth Engine layers
ndvi_vis = {
    'min': 0.0,
    'max': 0.8,
    'palette': ['white', 'yellow', 'green', 'darkgreen']
}

# Function to add EE image to Folium
def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

### **5. Compiling the Interactive Map**

In [ ]:
# Fetch WorldPop Population Density Data (2020)
worldpop = ee.ImageCollection('WorldPop/GP/100m/pop') \
    .filterDate('2020-01-01', '2021-01-01') \
    .mean() \
    .clip(ee_bounds)

# Fetch Sentinel-5P NO2 Air Quality Data
s5p_no2 = ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2') \
    .filterBounds(ee_bounds) \
    .filterDate('2023-01-01', '2023-12-31') \
    .select('tropospheric_NO2_column_number_density') \
    .median() \
    .clip(ee_bounds)

# Define visualization parameters for the new layers
pop_vis = {
    'bands': ['population'],
    'min': 0.0,
    'max': 50.0,
    'palette': ['24126c', '1fff4f', 'd4ff50']
}

no2_vis = {
    'min': 0,
    'max': 0.0002,
    'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

# Initialize map with Esri World Street Map tiles
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Source: Esri'
)

# 1. Add Andheri Boundary
folium.GeoJson(
    andheri_boundary,
    name="Andheri Boundary",
    style_function=lambda x: {'fillColor': 'none', 'color': 'black', 'weight': 2}
).add_to(m)

# 2. Add Earth Engine Layers
m.add_ee_layer(ndvi, ndvi_vis, 'NDVI (Green Spaces)')
m.add_ee_layer(worldpop, pop_vis, 'Population Density (WorldPop)')
m.add_ee_layer(s5p_no2, no2_vis, 'NO2 Density (Sentinel-5P)')

# 3. Add Amenity Markers
for idx, row in amenities_points.iterrows():
    raw_type = row.get('amenity') or row.get('leisure')
    # Safeguard against NaN/null values
    amenity_type = str(raw_type) if hasattr(raw_type, 'title') or (isinstance(raw_type, str) and raw_type != 'nan') else 'Amenity'
    color = 'red' if amenity_type == 'hospital' else 'blue' if amenity_type == 'school' else 'green'

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        popup=f"{amenity_type.title()}: {row.get('name', 'Unknown')}",
        color=color,
        fill=True,
        fill_color=color
    ).add_to(m)

# Add Layer Control to toggle layers on and off
folium.LayerControl().add_to(m)
m

In [ ]:
m.save("andheri_amenities_map.html")

### **Interpretation**
The final compiled interactive map provides a multi-layered spatial view of Andheri, Mumbai. Specifically, it does the following:

* **Environmental & Demographic Overlays:** The map integrates raster layers processed via Google Earth Engine, allowing users to view vegetation health (NDVI from Sentinel-2), population density (from WorldPop), and air quality/NO2 levels (from Sentinel-5P).   

*  **Urban Amenities**: It plots 130 fetched service locations as color-coded circle markers: red for hospitals, blue for schools, and green for parks or other amenities.   

*  **Interactivity:** A layer control feature is added at the end, allowing the user to toggle the Andheri boundary, environmental overlays, and amenity markers on and off to analyze specific spatial relationships.   

### **Conclusion**

The notebook successfully combines OpenStreetMap infrastructure data with Google Earth Engine environmental datasets to create a comprehensive spatial analysis tool for Andheri. By mapping 1940 road network nodes, 4286 street edges, and 130 key urban amenities alongside population and pollution metrics, the analysis allows users to visually assess how accessible crucial services like hospitals and schools are to densely populated areas, as well as how they intersect with local air quality and green spaces.

### **Colab File:** https://colab.research.google.com/drive/1FvwmruobdxJd9AYQNgKV78UBeCtn244i?usp=sharing